# 3D Perception Pipeline — Colab Setup

Run cells **top to bottom** once per session.

**Runtime:** GPU → A100  
**Estimated time:** ~15 minutes

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/perception_project'
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)
print('Drive mounted. Project root:', DRIVE_ROOT)

## 2. Clone the repo

In [ ]:
import os
REPO_URL = 'https://github.com/adityashah1603/3D-perception-pipeline.git'
REPO_DIR = '/content/3D-perception-pipeline'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull')

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

## 3. Install dependencies

~10 minutes on first run.

In [ ]:
import torch
cuda_tag  = 'cu' + torch.version.cuda.replace('.', '')[:3]
torch_tag = 'torch' + torch.__version__.split('+')[0]
print(f'CUDA tag: {cuda_tag}  |  Torch tag: {torch_tag}')

In [ ]:
# Install Shapely binary wheel first to avoid build-from-source errors
import subprocess
subprocess.run(['pip', 'install', 'shapely', '--upgrade', '--only-binary=shapely', '-q'], check=True)

# mmcv pre-built wheel matched to CUDA + torch
subprocess.run([
    'pip', 'install', 'mmcv==2.1.0',
    '-f', f'https://download.openmmlab.com/mmcv/dist/{cuda_tag}/{torch_tag}/index.html',
    '-q'
], check=True)

# mmengine, mmdet, mmdet3d
subprocess.run(['pip', 'install', 'mmengine>=0.10.0', 'mmdet>=3.2.0', '-q'], check=True)
subprocess.run(['pip', 'install', 'mmdet3d==1.4.0', '-q'], check=True)

# nuScenes devkit and utilities
subprocess.run(['pip', 'install', 'nuscenes-devkit>=1.1.11', 'open3d', 'matplotlib',
                'opencv-python', 'pyyaml', 'tqdm', '-q'], check=True)

print('All dependencies installed.')

In [ ]:
import mmdet3d, torch
print('mmdet3d version:', mmdet3d.__version__)
print('CUDA available :', torch.cuda.is_available())
print('GPU            :', torch.cuda.get_device_name(0))

## 4. Extract nuScenes v1.0-mini

Upload `v1.0-mini.tgz` to Drive at `MyDrive/perception_project/data/v1.0-mini.tgz` first.

In [ ]:
import os
DATA_DIR  = '/content/3D-perception-pipeline/data/nuscenes'
DRIVE_TGZ = f'{DRIVE_ROOT}/data/v1.0-mini.tgz'

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(DRIVE_TGZ):
    raise FileNotFoundError(f'Upload v1.0-mini.tgz to Drive first:\n  {DRIVE_TGZ}')

if not os.path.exists(f'{DATA_DIR}/v1.0-mini'):
    print('Extracting nuScenes mini ...')
    os.system(f'tar -xzf {DRIVE_TGZ} -C {DATA_DIR}')
    print('Done.')
else:
    print('Already extracted, skipping.')

os.system(f'ls {DATA_DIR}')

## 5. Generate mmdet3d annotation .pkl files

In [ ]:
import mmdet3d, os
MMDET3D_ROOT = os.path.dirname(mmdet3d.__file__)
PREP_SCRIPT  = os.path.join(MMDET3D_ROOT, '..', 'tools', 'create_data.py')

if not os.path.exists(PREP_SCRIPT):
    os.system('git clone --depth 1 -b v1.4.0 https://github.com/open-mmlab/mmdetection3d.git /content/mmdet3d_repo -q')
    PREP_SCRIPT = '/content/mmdet3d_repo/tools/create_data.py'

print('create_data.py:', PREP_SCRIPT)

In [ ]:
os.system(
    f'python {PREP_SCRIPT} nuscenes '
    f'--root-path {DATA_DIR} '
    f'--out-dir   {DATA_DIR} '
    f'--version   v1.0-mini '
    f'--extra-tag nuscenes'
)

print('Generated pkl files:')
os.system(f'ls {DATA_DIR}/*.pkl')

## 6. Verify data layout

In [ ]:
import os
expected = [
    'v1.0-mini',
    'maps',
    'samples/LIDAR_TOP',
    'sweeps/LIDAR_TOP',
    'nuscenes_infos_train.pkl',
    'nuscenes_infos_val.pkl',
]

all_ok = True
for item in expected:
    exists = os.path.exists(os.path.join(DATA_DIR, item))
    print(f'  {"OK" if exists else "MISSING"}  {item}')
    if not exists:
        all_ok = False

print()
print('Data layout OK!' if all_ok else 'Some items missing — check steps above.')

## 7. Quick nuScenes sanity check

In [ ]:
import numpy as np
from nuscenes.nuscenes import NuScenes

nusc = NuScenes(version='v1.0-mini', dataroot=DATA_DIR, verbose=False)
print(f'Scenes  : {len(nusc.scene)}')
print(f'Samples : {len(nusc.sample)}')

sample   = nusc.sample[0]
lidar_sd = nusc.get('sample_data', sample['data']['LIDAR_TOP'])
pcd_path = os.path.join(DATA_DIR, lidar_sd['filename'])
pts      = np.fromfile(pcd_path, dtype=np.float32).reshape(-1, 5)
print(f'First point cloud: {pts.shape}  path: {lidar_sd["filename"]}')

## 8. Train CenterPoint

In [ ]:
os.system('python tools/train.py --config configs/centerpoint.yaml')

## 9. Evaluate

In [ ]:
CKPT = 'work_dirs/centerpoint/epoch_20.pth'
os.system(f'python tools/evaluate.py --config configs/centerpoint.yaml --checkpoint {CKPT}')